# Hierarchical / Multi-Resolution GraphKoopman

This notebook uses the **power-user** module `koopman_graph.hierarchical`:

- Pool a fine grid with PyG **TopK** (optional `pooling="sag"`)
- Run a composed `GraphKoopmanModel` on the **coarse** graph
- Scatter-unpool predictions back to fine resolution

## Caveats (read first)

- This is **coarse-level forecasting with learned unpooling**, not
  physics-augmented spatiotemporal **super-resolution**.
- P-K-GCN (Zhang et al., 2026, arXiv:2606.19303) is a related *coarse→fine*
  precedent in a different problem setting — cite carefully; do not equate
  the APIs.
- Forecasting only at the coarsest level can miss fine-scale structure.
  Inspect both `resolution="coarse"` and `resolution="fine"`.
- `pool_ratios=(1.0,)` keeps all nodes (identity reduction) as a sanity check
  against a flat model.
- Classes are **not** on the root `koopman_graph` façade.


In [ ]:
import warnings

from tqdm.std import TqdmWarning

warnings.filterwarnings("ignore", category=TqdmWarning)

import os
import time

import matplotlib

if os.environ.get("PYTEST_CURRENT_TEST"):
    matplotlib.use("Agg")

import matplotlib.pyplot as plt
import torch

try:
    from IPython import get_ipython

    if get_ipython() is not None and not os.environ.get("PYTEST_CURRENT_TEST"):
        get_ipython().run_line_magic("matplotlib", "inline")
except (ImportError, NameError):
    pass

from koopman_graph import GNNDecoder, GNNEncoder, GraphKoopmanModel
from koopman_graph.datasets import GridDynamicGraphBenchmark
from koopman_graph.hierarchical import HierarchicalGraphKoopmanModel
from koopman_graph.metrics import evaluate_forecast

# Quick-run defaults; increase N / epochs for a fuller demo.
CI = bool(os.environ.get("PYTEST_CURRENT_TEST"))
N_ROWS = 8 if CI else 16
N_COLS = 8 if CI else 16
EPOCHS = 8 if CI else 40
STEPS = 5 if CI else 10
print(f"grid={N_ROWS}x{N_COLS} nodes={N_ROWS * N_COLS} epochs={EPOCHS}")


## Data: Laplacian diffusion on a 2D grid


In [ ]:
sequence = GridDynamicGraphBenchmark.generate(
    num_rows=N_ROWS,
    num_cols=N_COLS,
    num_timesteps=40 if CI else 60,
    in_channels=1,
    diffusion_rate=0.08,
    decay_rate=0.99,
    noise_std=0.0,
    seed=0,
)
print(sequence[0])
print(f"num_nodes={sequence.num_nodes}")


## Flat baseline vs hierarchical (TopK ratio 0.5)


In [ ]:
def make_model(seed: int = 0) -> GraphKoopmanModel:
    torch.manual_seed(seed)
    return GraphKoopmanModel(
        encoder=GNNEncoder(in_channels=1, hidden_channels=16, latent_dim=8),
        decoder=GNNDecoder(latent_dim=8, hidden_channels=16, out_channels=1),
        latent_dim=8,
        time_step=0.1,
    )


flat = make_model(seed=0)
t0 = time.perf_counter()
flat.fit(sequence, epochs=EPOCHS, lr=5e-3)
flat_fit_s = time.perf_counter() - t0

hier = HierarchicalGraphKoopmanModel(make_model(seed=0), pool_ratios=(0.5,))
t0 = time.perf_counter()
hier.fit(sequence, epochs=EPOCHS, lr=5e-3, unpool_epochs=max(3, EPOCHS // 4))
hier_fit_s = time.perf_counter() - t0

print(f"flat fit wall-clock:  {flat_fit_s:.2f}s")
print(f"hier fit wall-clock:  {hier_fit_s:.2f}s")


In [ ]:
origin = 0
horizon = STEPS

t0 = time.perf_counter()
flat_preds = flat.predict(sequence[origin], steps=horizon)
flat_pred_s = time.perf_counter() - t0

t0 = time.perf_counter()
hier_fine = hier.predict(sequence[origin], steps=horizon, resolution="fine")
hier_pred_s = time.perf_counter() - t0
hier_coarse = hier.predict(sequence[origin], steps=horizon, resolution="coarse")

flat_eval = evaluate_forecast(
    flat, sequence, horizons=(horizon,), start_indices=[origin]
)
hier_eval = evaluate_forecast(
    hier, sequence, horizons=(horizon,), start_indices=[origin]
)

print(f"flat predict wall-clock: {flat_pred_s * 1e3:.1f} ms")
print(f"hier predict wall-clock: {hier_pred_s * 1e3:.1f} ms")
print(f"flat RMSE:  {flat_eval.aggregate_rmse:.4f}")
print(f"hier RMSE:  {hier_eval.aggregate_rmse:.4f}")
print(
    f"coarse nodes: {hier_coarse[0].x.shape[0]}  "
    f"fine nodes: {hier_fine[0].x.shape[0]}"
)
_ = flat_preds  # keep for interactive inspection


## Wall-clock vs RMSE summary

On large grids the coarse latent advance is cheaper per step; unpooling
restores the fine node count for evaluation. Expect a **tradeoff**: hierarchical
forecasts may increase RMSE relative to a matched flat model while reducing
compute on the Koopman rollout. Results below are teaching-scale, not a claim
of Pareto optimality.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].bar(["flat", "hier"], [flat_fit_s, hier_fit_s], color=["#4C72B0", "#55A868"])
ax[0].set_ylabel("fit wall-clock (s)")
ax[0].set_title("Training time")
ax[1].bar(
    ["flat", "hier"],
    [flat_eval.aggregate_rmse, hier_eval.aggregate_rmse],
    color=["#4C72B0", "#55A868"],
)
ax[1].set_ylabel("RMSE")
ax[1].set_title(f"{horizon}-step forecast RMSE")
fig.tight_layout()
plt.show()


## No-op sanity: `pool_ratios=(1.0,)`

PyG would treat float `1.0` as "keep 1 node"; KoopmanGraph special-cases
full retention so this path keeps all nodes and should stay close to flat
capacity (identity unpool refine).


In [ ]:
noop = HierarchicalGraphKoopmanModel(make_model(seed=1), pool_ratios=(1.0,))
noop.fit(sequence, epochs=max(3, EPOCHS // 2), lr=5e-3, unpool_epochs=2)
coarse0, steps = noop.pool_down(sequence[0])
assert coarse0.x.shape[0] == sequence.num_nodes
print("noop coarse nodes:", coarse0.x.shape[0], "== fine", sequence.num_nodes)


## Takeaways

1. Import hierarchical APIs from `koopman_graph.hierarchical` (power-user).
2. Default pooling is TopK; pass `pooling="sag"` for SAGPooling.
3. Use `predict(..., resolution="coarse"|"fine")` to inspect multi-resolution
   outputs; graph-operator spectra use the **pooled** topology.
4. Do not market this wrapper as P-K-GCN super-resolution.
